# Lab W7: LLM-Assisted Feature Development

## Pre-flight Checklist

> [!IMPORTANT]
> Konsep yang ditandai (§) merujuk ke `07_W7_Text_Transformers_Repo_Adoption.md` §2 (AI Tools sebagai Pendukung).

**Yang Anda butuhkan sebelum mulai:**
- Bab W7 §2 sudah dibaca (verifikasi rule, synthesis rule 2 sumber, AI untuk non-kode).
- Akses ke salah satu LLM tool: Claude, ChatGPT, atau Copilot.
- Lab W2 selesai (training pipeline jalan, baseline ada).
- Familiar dengan paper Mixup (Zhang et al. 2018, arXiv:1710.09412) atau setidaknya konsep dasarnya.

**Yang akan Anda hasilkan di akhir lab:**
- Pseudocode mixup_batch yang Anda tulis sendiri **sebelum** minta ke LLM (proof Anda sudah berpikir).
- Prompt LLM verbatim yang dipakai (untuk traceability).
- Output LLM verbatim sebagai baseline modifikasi.
- 4 sanity tests: alpha=0 ≡ pass-through, lam ∈ [0,1], shape preserved, mixup_criterion benar.
- Eksperimen baseline vs mixup (2 seed × 15 epoch).
- `docs/llm_log.md` updated dengan entri lab.

**Resource:**
- **Hardware:** GPU dianjurkan untuk eksperimen multi-seed (CPU 2 seed × 15 epoch ~1-2 jam).
- **Estimasi waktu kerja:** 3-4 jam termasuk pseudocode, prompting, eksperimen.

**Pendamping:** Bab W7 §2.

## Skenario
PI meminta: *"tambahkan mixup augmentation ke training loop, lalu bandingkan dengan baseline"*. Anda belum pernah mengimplementasi Mixup. Gunakan LLM untuk *mempercepat* implementasi, bukan menggantikan verifikasi.

## Tujuan
1. Implementasikan `mixup_batch` dengan bantuan LLM menggunakan 5-tahap protokol.
2. Verifikasi output dengan sanity test (alpha=0 ≡ no augmentation).
3. Jalankan baseline vs mixup (2 seed, 15 epoch).
4. Catat seluruh proses di `docs/llm_log.md`.

## Checklist
- [ ] Pseudocode manual ditulis SEBELUM minta ke LLM (bukan setelahnya).
- [ ] Prompt yang dipakai disalin ke notebook (bukan rekonstruksi dari memori).
- [ ] Output LLM *sebelum* modifikasi disalin sebagai komentar/markdown.
- [ ] Sanity test: `mixup_batch(x, y, alpha=0.0)` ekuivalen dengan pass-through.
- [ ] Sanity test: `mixup_batch(x, y, alpha=1.0)` menghasilkan nilai lam ∈ [0, 1].
- [ ] Hasil mixup vs baseline dilaporkan dengan 2 seed.
- [ ] `docs/llm_log.md` diupdate dengan entri lab ini.

## 1. Pseudocode manual SEBELUM minta ke LLM

**Ini wajib dilakukan dulu.** Tulis algoritma dalam bahasa kamu sendiri, tanpa kode. Tujuannya: kamu bisa mendeteksi jika LLM memberikan implementasi yang berbeda dari yang kamu maksud.

Referensi: Zhang et al. 2018, "mixup: Beyond Empirical Risk Minimization" (arXiv:1710.09412)

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Di Google Colab: clone repo terlebih dahulu, lalu cd ke template_repo/
#   git clone <url-repo> && cd template_repo && pip install -e .
# Di environment lokal: jalankan dari folder notebooks/
import sys, os
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Root:", _root)

### Pseudocode Mixup (isi sendiri)

```
Fungsi mixup_batch(x, y, alpha):
  1. [tulis langkah 1]
  2. [tulis langkah 2]
  3. [dst.]
  Output: [apa yang dikembalikan]
```

## 2. Prompt yang dipakai ke LLM

Tempel prompt *persis* yang kamu kirimkan. Bukan rekonstruksi. Bukan rangkuman. Copy-paste.
Ini penting untuk traceability — jika implementasi ada bug, kita bisa melihat apakah bug berasal dari prompt yang kurang jelas.

```
[TEMPEL PROMPT DI SINI]
```

## 3. Output LLM (verbatim, sebelum modifikasi)

Tempel output asli LLM sebagai komentar kode atau markdown. Ini adalah *baseline* untuk membandingkan perubahan yang kamu buat.

```python
# [OUTPUT LLM VERBATIM DI SINI]
# (salin seluruh fungsi yang diberikan LLM, tanpa mengubah satu pun karakter)
```

## 4. Implementasi yang sudah diverifikasi dan dimodifikasi

In [ ]:
import torch
import numpy as np

def mixup_batch(
    x: torch.Tensor,
    y: torch.Tensor,
    alpha: float = 0.2
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """Mixup augmentation (Zhang et al. 2018).

    Returns:
        mixed_x: augmented batch
        y_a: original labels
        y_b: shuffled labels
        lam: mixing coefficient (0 = all y_b, 1 = all y_a)
    """
    if alpha <= 0.0:
        return x, y, y, 1.0

    # Sample lam from Beta distribution
    lam = float(np.random.beta(alpha, alpha))

    # Random permutation of the batch
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    # Mix inputs
    mixed_x = lam * x + (1 - lam) * x[index]

    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute mixup loss: lam * L(pred, y_a) + (1-lam) * L(pred, y_b)."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('Fungsi mixup_batch dan mixup_criterion didefinisikan.')

## 5. Sanity tests (wajib sebelum eksperimen)

In [ ]:
torch.manual_seed(0)

x = torch.randn(8, 3, 32, 32)
y = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7])

# Test 1: alpha=0 harus pass-through
mx, ya, yb, lam = mixup_batch(x, y, alpha=0.0)
assert torch.allclose(mx, x), 'alpha=0 harus mengembalikan x tanpa perubahan'
assert lam == 1.0, 'alpha=0 harus lam=1.0'
assert (ya == yb).all(), 'alpha=0: y_a harus sama dengan y_b'
print('✓ Test 1 lulus: alpha=0 → pass-through')

# Test 2: alpha>0, lam harus dalam [0, 1]
lams = [mixup_batch(x, y, alpha=1.0)[3] for _ in range(100)]
assert all(0 <= l <= 1 for l in lams), 'lam harus selalu dalam [0, 1]'
print(f'✓ Test 2 lulus: lam selalu ∈ [0,1]. Mean lam: {np.mean(lams):.3f} (harapan ≈ 0.5 untuk alpha=1)')

# Test 3: mixed_x harus berbeda dari x (kecuali kebetulan lam=1)
mx, _, _, lam = mixup_batch(x, y, alpha=0.5)
assert mx.shape == x.shape, f'Shape harus sama. Got: {mx.shape}'
print(f'✓ Test 3 lulus: output shape benar. lam={lam:.3f}')

# Test 4: loss dengan mixup criterion
model_logits = torch.randn(8, 10)
criterion = torch.nn.CrossEntropyLoss()
loss_normal  = criterion(model_logits, y).item()
mx, ya, yb, lam = mixup_batch(x, y, alpha=0.5)
loss_mixup = mixup_criterion(criterion, model_logits, ya, yb, lam).item()
print(f'✓ Test 4 lulus: loss_normal={loss_normal:.4f}, loss_mixup={loss_mixup:.4f}')

print('\nSemua sanity test lulus. Lanjutkan ke eksperimen.')

from src.models import build_model
from src.losses import build_loss
from src.data import build_loaders
from src.utils import load_config, set_seed
from src.train import build_optimizer, build_scheduler, evaluate

import time

def train_with_mixup(cfg, seed, use_mixup=False, alpha=0.2, max_epochs=15):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    cfg = {**cfg, 'train': {**cfg['train'], 'epochs': max_epochs}}

    train_loader, val_loader, _ = build_loaders(cfg['data'])
    model = build_model(cfg['model']).to(device)
    loss_fn = build_loss(cfg['loss']).to(device)
    optimizer = build_optimizer(list(model.parameters()), cfg['optim'])
    scheduler = build_scheduler(optimizer, cfg.get('scheduler'), max_epochs)

    best_val = 0.0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = torch.as_tensor(y).long().view(-1).to(device)

            if use_mixup:
                x, y_a, y_b, lam = mixup_batch(x, y, alpha=alpha)
                logits = model(x)
                loss = mixup_criterion(loss_fn, logits, y_a, y_b, lam)
            else:
                logits = model(x)
                loss = loss_fn(logits, y)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * x.size(0)
            total += x.size(0)

        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        if scheduler:
            scheduler.step()

        best_val = max(best_val, val_acc)
        history.append({'epoch': epoch, 'val_acc': val_acc, 'train_loss': loss_sum / total})

    return best_val, history


cfg = load_config('../configs/baseline.yaml')
seeds = [42, 123]
results = {'baseline': [], 'mixup': []}

for seed in seeds:
    print(f'Seed {seed} - baseline...', end=' ', flush=True)
    t0 = time.time()
    best_val, _ = train_with_mixup(cfg, seed, use_mixup=False)
    results['baseline'].append(best_val)
    print(f'val_acc={best_val:.4f} ({time.time()-t0:.0f}s)')

    print(f'Seed {seed} - mixup...  ', end=' ', flush=True)
    t0 = time.time()
    best_val, _ = train_with_mixup(cfg, seed, use_mixup=True, alpha=0.2)
    results['mixup'].append(best_val)
    print(f'val_acc={best_val:.4f} ({time.time()-t0:.0f}s)')

In [ ]:
from src.models import build_model
from src.losses import build_loss
from src.data import build_loaders
from src.utils import load_config, set_seed
from src.train import build_optimizer, build_scheduler, evaluate

import time

def train_with_mixup(cfg, seed, use_mixup=False, alpha=0.2, max_epochs=15):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Override epochs untuk eksperimen singkat
    cfg = {**cfg, 'train': {**cfg['train'], 'epochs': max_epochs}}

    train_loader, val_loader, _ = build_loaders(cfg['data'])
    model = build_model(cfg['model']).to(device)
    loss_fn = build_loss(cfg['loss']).to(device)
    optimizer = build_optimizer(list(model.parameters()), cfg['optim'])
    scheduler = build_scheduler(optimizer, cfg.get('scheduler'), max_epochs)

    best_val = 0.0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = torch.as_tensor(y).long().view(-1).to(device)

            if use_mixup:
                x, y_a, y_b, lam = mixup_batch(x, y, alpha=alpha)
                logits = model(x)
                loss = mixup_criterion(loss_fn, logits, y_a, y_b, lam)
            else:
                logits = model(x)
                loss = loss_fn(logits, y)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * x.size(0)
            total += x.size(0)

        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        if scheduler:
            scheduler.step()

        best_val = max(best_val, val_acc)
        history.append({'epoch': epoch, 'val_acc': val_acc, 'train_loss': loss_sum / total})

    return best_val, history


cfg = load_config('configs/baseline.yaml')
seeds = [42, 123]
results = {'baseline': [], 'mixup': []}

for seed in seeds:
    print(f'Seed {seed} - baseline...', end=' ', flush=True)
    t0 = time.time()
    best_val, _ = train_with_mixup(cfg, seed, use_mixup=False)
    results['baseline'].append(best_val)
    print(f'val_acc={best_val:.4f} ({time.time()-t0:.0f}s)')

    print(f'Seed {seed} - mixup...  ', end=' ', flush=True)
    t0 = time.time()
    best_val, _ = train_with_mixup(cfg, seed, use_mixup=True, alpha=0.2)
    results['mixup'].append(best_val)
    print(f'val_acc={best_val:.4f} ({time.time()-t0:.0f}s)')

In [ ]:
import matplotlib.pyplot as plt

baseline_arr = np.array(results['baseline'])
mixup_arr    = np.array(results['mixup'])

print('=== Hasil Baseline vs Mixup (15 epoch, 2 seed) ===')
print(f'Baseline: {baseline_arr * 100}  →  μ={baseline_arr.mean()*100:.2f}%, σ={baseline_arr.std()*100:.2f}%')
print(f'Mixup:    {mixup_arr * 100}  →  μ={mixup_arr.mean()*100:.2f}%, σ={mixup_arr.std()*100:.2f}%')
delta = (mixup_arr.mean() - baseline_arr.mean()) * 100
print(f'Δ(mixup - baseline): {delta:+.2f}%')

fig, ax = plt.subplots(figsize=(6, 4))
conditions = ['Baseline', 'Mixup (α=0.2)']
means = [baseline_arr.mean() * 100, mixup_arr.mean() * 100]
stds  = [baseline_arr.std() * 100, mixup_arr.std() * 100]
bars = ax.bar(conditions, means, yerr=stds, capsize=8, color=['steelblue', 'tomato'],
              alpha=0.85, edgecolor='white')
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.3,
            f'{mean:.1f}±{std:.1f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Best Val Accuracy (%) — mean ± std, 2 seed')
ax.set_title('Lab 5: Baseline vs Mixup (15 epoch)')
ax.set_ylim(max(0, min(means) - max(stds) - 5), max(means) + max(stds) + 5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../experiments/plots/lab5_mixup.png', dpi=120, bbox_inches='tight')
plt.show()

from pathlib import Path
from datetime import date

log_path = Path('../docs/llm_log.md')
log_path.parent.mkdir(exist_ok=True)

entry_template = f"""
---

## {date.today()} - Lab 5: Mixup Augmentation

**Tool:** [Claude / ChatGPT / Copilot — pilih yang dipakai]  
**Goal:** Implementasi mixup_batch untuk training loop CIFAR-10  

**Prompt:**
```
[tempel di sini]
```

**Apa yang diubah dari output LLM:**
- [perubahan 1]
- [perubahan 2]

**Sanity tests yang dijalankan:**
- alpha=0 → pass-through: LULUS/GAGAL
- lam ∈ [0,1]: LULUS/GAGAL

**Catatan:**
[apa yang paling berguna dari LLM, apa yang perlu kamu perbaiki sendiri]
"""

if not log_path.exists():
    log_path.write_text('# LLM Interaction Log\n' + entry_template)
    print(f'Dibuat: {log_path}')
else:
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(entry_template)
    print(f'Diupdate: {log_path}')

In [ ]:
from pathlib import Path
from datetime import date

log_path = Path('docs/llm_log.md')
log_path.parent.mkdir(exist_ok=True)

entry_template = f"""
---

## {date.today()} - Lab 5: Mixup Augmentation

**Tool:** [Claude / ChatGPT / Copilot — pilih yang dipakai]  
**Goal:** Implementasi mixup_batch untuk training loop CIFAR-10  

**Prompt:**
```
[tempel di sini]
```

**Apa yang diubah dari output LLM:**
- [perubahan 1]
- [perubahan 2]

**Sanity tests yang dijalankan:**
- alpha=0 → pass-through: LULUS/GAGAL
- lam ∈ [0,1]: LULUS/GAGAL

**Catatan:**
[apa yang paling berguna dari LLM, apa yang perlu kamu perbaiki sendiri]
"""

if not log_path.exists():
    log_path.write_text('# LLM Interaction Log\n' + entry_template)
    print(f'Dibuat: {log_path}')
else:
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(entry_template)
    print(f'Diupdate: {log_path}')

## 8. Refleksi

1. Apa bagian output LLM yang paling berbeda dari pseudocode-mu? Apakah perbedaannya adalah *bug*, *perbedaan desain*, atau *hal yang tidak kamu pikirkan sebelumnya*?

2. Jika sanity test `alpha=0` gagal, apa yang itu beritahu tentang implementasi? Apa kemungkinan penyebabnya?

3. Mixup biasanya membantu regularisasi. Jika hasilmu menunjukkan mixup *tidak* membantu atau bahkan menurunkan akurasi pada 15 epoch, apa hipotesismu? Eksperimen apa yang akan menjawabnya?

### Jawaban Refleksi

**1. Perbedaan terbesar dari pseudocode:**
> *[tulis di sini]*

**2. Jika alpha=0 gagal:**
> *[tulis di sini]*

**3. Jika mixup tidak membantu:**
> *[tulis di sini]*